# Part 05: TROPOMI Methane Panel Regression Analysis

**Scientific Question:** Does lake ice-off timing predict TROPOMI methane concentrations? Specifically, as lakes thaw within a TROPOMI pixel, does CH4 increase?

**Hypothesis:** As lakes thaw within a TROPOMI pixel, cumulative thawed lake area/perimeter should predict CH4 variation, controlling for pixel and time fixed effects.

**Approach:**
1. **Data Preparation:** Load lake phenology, assign lakes to 0.2° × 0.2° TROPOMI grid, compute pixel-level aggregates
2. **Cumulative Thaw Time Series:** For each pixel-week, calculate cumulative thawed area and perimeter
3. **TROPOMI CH4 with QA Filtering:** Extract weekly CH4 with qa_value > 0.5 filter
4. **Panel Regression:** CH4_it = β₁(thawed_area_it) + pixel_FE_i + week_FE_t + ε_it
5. **Robustness Checks:** Test area vs perimeter, lagged effects, seasonal restrictions

**Data Sources:**
- Ice phenology: `gs://wustl-eeps-geospatial/thermokarst_lakes/results/alaska_lakes_ice_phenology_2019-2023.csv`
- TROPOMI CH4: `COPERNICUS/S5P/OFFL/L3_CH4` (Google Earth Engine)

**Configuration:**
- Grid resolution: 0.2° × 0.2° (~7.6 km at 70°N) - matches TROPOMI native resolution
- Temporal resolution: Weekly (ISO weeks)
- Pixel filter: Minimum 0.5 km² total lake area per pixel
- Study period: March-October, 2019-2023 (~35 weeks × 5 years)

In [2]:
# Imports and Configuration
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import box
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import ee
import warnings
from datetime import datetime, timedelta

# Panel regression
from linearmodels.panel import PanelOLS

# Specific warning suppressions
warnings.filterwarnings('ignore', category=FutureWarning, module='google.api_core')
warnings.filterwarnings('ignore', category=FutureWarning, module='pyproj')
warnings.filterwarnings('ignore', message='.*Shapely GEOS.*')
warnings.filterwarnings('ignore', message='.*httplib2 transport.*')

# Initialize Earth Engine
try:
    ee.Initialize(project='eeps-geospatial')
    print("Earth Engine initialized successfully")
except ee.EEException:
    ee.Authenticate()
    ee.Initialize(project='eeps-geospatial')
    print("Earth Engine authenticated and initialized")

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

# Configuration
BUCKET_NAME = 'wustl-eeps-geospatial'
RESULTS_GCS = f'gs://{BUCKET_NAME}/thermokarst_lakes/results'
FIGURES_DIR = './figures'

# Study area bounds
LAT_MIN, LAT_MAX = 68.0, 72.0
LON_MIN, LON_MAX = -161.0, -145.0

# Grid resolution (degrees) - 0.2° matches TROPOMI native ~7km resolution
GRID_RES = 0.2

# Minimum total lake area per pixel (km²)
MIN_LAKE_AREA_KM2 = 0.5

# Years to analyze
YEARS = [2019, 2020, 2021, 2022, 2023]

# Weeks to analyze (March-October, approximately ISO weeks 9-44)
# Week 9 ~ early March, Week 44 ~ late October
WEEK_START = 9
WEEK_END = 44

# Create study area geometry
study_area = ee.Geometry.Rectangle([LON_MIN, LAT_MIN, LON_MAX, LAT_MAX])

print(f"\nStudy Area: {LAT_MIN}-{LAT_MAX}°N, {LON_MIN}-{LON_MAX}°E")
print(f"Grid Resolution: {GRID_RES}° (~{GRID_RES * 111 * np.cos(np.radians(70)):.1f} km at 70°N)")
print(f"Temporal Resolution: Weekly (ISO weeks {WEEK_START}-{WEEK_END})")
print(f"Pixel Filter: Total lake area >= {MIN_LAKE_AREA_KM2} km²")
print(f"Years: {YEARS}")

ModuleNotFoundError: No module named 'linearmodels'

## Part 1: Load Ice Phenology Data

Load the ice phenology results from Notebook 03 and compute summary statistics.

In [3]:
# Load ice phenology data from GCS
print("Loading ice phenology data...")

phenology_path = f'{RESULTS_GCS}/alaska_lakes_ice_phenology_2019-2023.csv'

try:
    phenology_df = pd.read_csv(phenology_path)
    print(f"Loaded {len(phenology_df):,} lake-year records from GCS")
except FileNotFoundError:
    # Fallback to local file if GCS export hasn't completed
    print("GCS file not found, trying local fallback...")
    local_path = './first_550/alaska_lakes_ice_events_multisensor_2024.csv'
    phenology_df = pd.read_csv(local_path)
    # Rename columns to match expected format
    phenology_df = phenology_df.rename(columns={
        'first_no_ice_doy': 'ice_off_doy',
        'first_full_ice_doy': 'ice_on_doy'
    })
    phenology_df['year'] = 2024
    print(f"Loaded {len(phenology_df):,} records from local file")

# Display summary
print(f"\nUnique lakes: {phenology_df['lake_id'].nunique():,}")
print(f"Years: {sorted(phenology_df['year'].unique())}")

# Show columns
print(f"\nColumns: {list(phenology_df.columns)}")

# Display sample
display(phenology_df.head())

Loading ice phenology data...


NameError: name 'RESULTS_GCS' is not defined

,lake_id,year,centroid_lon,centroid_lat,area_km2,circularity,sdi,convexity,ice_off_date,ice_off_doy,ice_off_conf,ice_on_date,ice_on_doy,ice_on_conf,ice_free_days
0,6316,2019,-162.017627,70.211427,0.340748,0.19257,2.278794,0.77219,2019-06-20,171.0,medium,2019-10-30,303.0,low,132.0
1,6316,2020,-162.017627,70.211427,0.340748,0.19257,2.278794,0.77219,2020-06-18,170.0,low,2020-09-30,274.0,low,104.0
2,6316,2021,-162.017627,70.211427,0.340748,0.19257,2.278794,0.77219,2021-06-25,176.0,low,2021-09-17,260.0,low,84.0
3,6316,2022,-162.017627,70.211427,0.340748,0.19257,2.278794,0.77219,2022-06-28,179.0,low,2022-09-20,263.0,low,84.0
4,6316,2023,-162.017627,70.211427,0.340748,0.19257,2.278794,0.77219,2023-07-05,186.0,low,2023-11-14,318.0,low,132.0


In [ ]:
# Filter to complete records (both ice-off and ice-on detected)
complete_phenology = phenology_df[
    phenology_df['ice_off_doy'].notna() & 
    phenology_df['ice_on_doy'].notna()
].copy()

print(f"Complete records: {len(complete_phenology):,} / {len(phenology_df):,} ({100*len(complete_phenology)/len(phenology_df):.1f}%)")

# Summary statistics
print(f"\nPhenology Statistics:")
print(f"  Ice-off DOY: {complete_phenology['ice_off_doy'].mean():.1f} ± {complete_phenology['ice_off_doy'].std():.1f}")
print(f"  Ice-on DOY:  {complete_phenology['ice_on_doy'].mean():.1f} ± {complete_phenology['ice_on_doy'].std():.1f}")
print(f"  Ice-free days: {complete_phenology['ice_free_days'].mean():.1f} ± {complete_phenology['ice_free_days'].std():.1f}")

## Part 2: Create Analysis Grid

Create a 0.2° × 0.2° fixed lat/lon grid (~7.6 km at 70°N) matching TROPOMI native resolution, and assign lakes to grid cells.

In [ ]:
# Create grid edges
lat_edges = np.arange(LAT_MIN, LAT_MAX + GRID_RES, GRID_RES)
lon_edges = np.arange(LON_MIN, LON_MAX + GRID_RES, GRID_RES)

print(f"Grid dimensions: {len(lat_edges)-1} rows × {len(lon_edges)-1} cols = {(len(lat_edges)-1) * (len(lon_edges)-1)} cells")
print(f"Latitude edges: {lat_edges[0]:.1f}° to {lat_edges[-1]:.1f}°")
print(f"Longitude edges: {lon_edges[0]:.1f}° to {lon_edges[-1]:.1f}°")

# Create grid cells as GeoDataFrame
grid_cells = []
cell_id = 0

for i in range(len(lat_edges) - 1):
    for j in range(len(lon_edges) - 1):
        lat_min = lat_edges[i]
        lat_max = lat_edges[i + 1]
        lon_min = lon_edges[j]
        lon_max = lon_edges[j + 1]
        
        cell = {
            'cell_id': cell_id,
            'lat_min': lat_min,
            'lat_max': lat_max,
            'lon_min': lon_min,
            'lon_max': lon_max,
            'lat_center': (lat_min + lat_max) / 2,
            'lon_center': (lon_min + lon_max) / 2,
            'geometry': box(lon_min, lat_min, lon_max, lat_max)
        }
        grid_cells.append(cell)
        cell_id += 1

grid_gdf = gpd.GeoDataFrame(grid_cells, crs='EPSG:4326')
print(f"\nCreated {len(grid_gdf)} grid cells")

In [ ]:
# Assign lakes to grid cells and compute pixel-level aggregates
print("Assigning lakes to grid cells...")

# Create Point geometries for lakes
lake_points = gpd.GeoDataFrame(
    complete_phenology,
    geometry=gpd.points_from_xy(complete_phenology['centroid_lon'], complete_phenology['centroid_lat']),
    crs='EPSG:4326'
)

# Spatial join to assign lakes to grid cells
lakes_with_cells = gpd.sjoin(lake_points, grid_gdf[['cell_id', 'geometry']], how='left', predicate='within')

# Compute pixel-level aggregates for each cell
# Note: sdi (Shoreline Development Index) can be used to compute perimeter from area
# perimeter = sdi * 2 * sqrt(pi * area)
lakes_with_cells['lake_perim_km'] = lakes_with_cells['sdi'] * 2 * np.sqrt(np.pi * lakes_with_cells['area_km2'])

cell_agg = lakes_with_cells.groupby('cell_id').agg({
    'lake_id': 'count',  # number of lakes
    'area_km2': 'sum',    # total lake area
    'lake_perim_km': 'sum'  # total lake perimeter
}).reset_index()
cell_agg.columns = ['cell_id', 'n_lakes', 'total_lake_area_km2', 'total_lake_perim_km']

# Merge with grid
grid_with_lakes = grid_gdf.merge(cell_agg, on='cell_id', how='left')
grid_with_lakes['n_lakes'] = grid_with_lakes['n_lakes'].fillna(0).astype(int)
grid_with_lakes['total_lake_area_km2'] = grid_with_lakes['total_lake_area_km2'].fillna(0)
grid_with_lakes['total_lake_perim_km'] = grid_with_lakes['total_lake_perim_km'].fillna(0)

# Filter to cells with minimum lake area (instead of minimum lake count)
active_cells = grid_with_lakes[grid_with_lakes['total_lake_area_km2'] >= MIN_LAKE_AREA_KM2].copy()

print(f"\nPixels with >= {MIN_LAKE_AREA_KM2} km² total lake area: {len(active_cells)} / {len(grid_gdf)}")
print(f"Total lakes in active pixels: {active_cells['n_lakes'].sum():.0f}")
print(f"Total lake area in active pixels: {active_cells['total_lake_area_km2'].sum():.1f} km²")
print(f"Total lake perimeter in active pixels: {active_cells['total_lake_perim_km'].sum():.1f} km")

# Display summary statistics
print(f"\nPixel-level statistics:")
print(f"  Lakes per pixel: {active_cells['n_lakes'].mean():.1f} ± {active_cells['n_lakes'].std():.1f}")
print(f"  Area per pixel: {active_cells['total_lake_area_km2'].mean():.2f} ± {active_cells['total_lake_area_km2'].std():.2f} km²")
print(f"  Perimeter per pixel: {active_cells['total_lake_perim_km'].mean():.2f} ± {active_cells['total_lake_perim_km'].std():.2f} km")

In [ ]:
# Visualize grid with lake area density
fig, ax = plt.subplots(figsize=(14, 10))

# Plot all grid cells (empty)
grid_gdf.boundary.plot(ax=ax, color='lightgray', linewidth=0.5)

# Plot active cells colored by total lake area
active_cells.plot(column='total_lake_area_km2', ax=ax, legend=True, 
                  cmap='YlOrRd', edgecolor='black', linewidth=0.5,
                  legend_kwds={'label': 'Total Lake Area (km²)', 'shrink': 0.6})

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Analysis Grid ({GRID_RES}° × {GRID_RES}°)\n{len(active_cells)} pixels with >= {MIN_LAKE_AREA_KM2} km² lake area')
ax.set_xlim(LON_MIN - 0.5, LON_MAX + 0.5)
ax.set_ylim(LAT_MIN - 0.2, LAT_MAX + 0.2)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/manuscript/ch4_analysis_grid.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: ch4_analysis_grid.png")

## Part 3: Build Cumulative Thaw Time Series

For each pixel-week, compute the cumulative thawed lake area and perimeter (lakes where ice_off_doy <= week midpoint DOY).

In [ ]:
# Build cumulative thaw time series for each pixel-year-week
print("Building cumulative thaw time series...")

def week_to_doy(year, week):
    """Convert ISO week to approximate day of year (midpoint of week)."""
    # Get the Monday of the ISO week
    jan4 = datetime(year, 1, 4)  # Jan 4 is always in week 1
    monday_week1 = jan4 - timedelta(days=jan4.weekday())
    target_monday = monday_week1 + timedelta(weeks=week - 1)
    # Return midpoint of week (Thursday = +3 days)
    midpoint = target_monday + timedelta(days=3)
    return midpoint.timetuple().tm_yday

# Get unique cell_ids from active_cells
active_cell_ids = set(active_cells['cell_id'].values)

# Prepare lake data with cell assignments
lake_data = lakes_with_cells[lakes_with_cells['cell_id'].isin(active_cell_ids)].copy()
lake_data['lake_perim_km'] = lake_data['sdi'] * 2 * np.sqrt(np.pi * lake_data['area_km2'])

print(f"Lakes in active pixels: {len(lake_data):,}")
print(f"Unique pixels: {lake_data['cell_id'].nunique()}")

# Build thaw time series records
thaw_records = []
total_combos = len(YEARS) * (WEEK_END - WEEK_START + 1) * len(active_cell_ids)
print(f"\nProcessing {len(YEARS)} years × {WEEK_END - WEEK_START + 1} weeks × {len(active_cell_ids)} pixels = {total_combos:,} combinations")

for year in YEARS:
    # Get lakes with ice_off data for this year
    year_lakes = lake_data[lake_data['year'] == year].copy()
    
    for week in range(WEEK_START, WEEK_END + 1):
        target_doy = week_to_doy(year, week)
        
        for cell_id in active_cell_ids:
            cell_lakes = year_lakes[year_lakes['cell_id'] == cell_id]
            
            if len(cell_lakes) == 0:
                continue
            
            # Get total lake area/perimeter for this cell
            total_area = cell_lakes['area_km2'].sum()
            total_perim = cell_lakes['lake_perim_km'].sum()
            
            # Compute thawed area/perimeter (lakes where ice_off_doy <= target_doy)
            thawed = cell_lakes[cell_lakes['ice_off_doy'] <= target_doy]
            thawed_area = thawed['area_km2'].sum() if len(thawed) > 0 else 0
            thawed_perim = thawed['lake_perim_km'].sum() if len(thawed) > 0 else 0
            n_thawed = len(thawed)
            
            # Fraction thawed
            frac_thawed = thawed_area / total_area if total_area > 0 else 0
            
            # Get cell centroid
            cell_info = active_cells[active_cells['cell_id'] == cell_id].iloc[0]
            
            thaw_records.append({
                'cell_id': int(cell_id),
                'year': year,
                'week': week,
                'doy': target_doy,
                'thawed_area_km2': thawed_area,
                'thawed_perim_km': thawed_perim,
                'frac_thawed': frac_thawed,
                'n_thawed_lakes': n_thawed,
                'total_area_km2': total_area,
                'total_perim_km': total_perim,
                'n_lakes': len(cell_lakes),
                'lat_center': cell_info['lat_center'],
                'lon_center': cell_info['lon_center']
            })

thaw_df = pd.DataFrame(thaw_records)
print(f"\nThaw time series: {len(thaw_df):,} pixel-week observations")
print(f"Unique pixels: {thaw_df['cell_id'].nunique()}")
print(f"Date range: week {thaw_df['week'].min()}-{thaw_df['week'].max()}, DOY {thaw_df['doy'].min()}-{thaw_df['doy'].max()}")

# Summary by year
print(f"\nObservations by year:")
for year in YEARS:
    n = len(thaw_df[thaw_df['year'] == year])
    print(f"  {year}: {n:,}")

display(thaw_df.head(10))

In [ ]:
# Visualize thaw progression
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Mean thaw progression by week
ax = axes[0]
thaw_by_week = thaw_df.groupby(['year', 'week']).agg({
    'frac_thawed': 'mean',
    'doy': 'first'
}).reset_index()

for year in YEARS:
    year_data = thaw_by_week[thaw_by_week['year'] == year]
    ax.plot(year_data['doy'], year_data['frac_thawed'], 'o-', label=str(year), alpha=0.7)

ax.set_xlabel('Day of Year')
ax.set_ylabel('Mean Fraction Thawed')
ax.set_title('Thaw Progression by Year')
ax.legend(title='Year')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

# Panel B: Distribution of thawed area
ax = axes[1]
# Focus on transition period (DOY 140-200)
transition = thaw_df[(thaw_df['doy'] >= 140) & (thaw_df['doy'] <= 200)]
ax.hist(transition['frac_thawed'], bins=50, edgecolor='black', alpha=0.7)
ax.set_xlabel('Fraction Thawed')
ax.set_ylabel('Count')
ax.set_title(f'Distribution of Thaw State\n(DOY 140-200, n={len(transition):,})')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/supplementary/ch4_thaw_progression.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: ch4_thaw_progression.png")

## Part 4: Extract TROPOMI CH4 Data (Weekly)

Extract weekly average CH4 column mixing ratios from Sentinel-5P TROPOMI for each grid cell.

**Note on Quality Filtering:**
- The L3 CH4 product (`COPERNICUS/S5P/OFFL/L3_CH4`) is already quality-filtered during L2→L3 conversion
- Google applies `CH4_column_volume_mixing_ratio_dry_air_validity > 50` via harpconvert
- We use the `_bias_corrected` band which is recommended for analysis
- Optional: Filter by uncertainty band for tighter quality control

**Approach:** Submit GEE export tasks to Cloud Storage, then load results.

In [ ]:
# Create grid FeatureCollection for GEE exports
def create_grid_feature_collection(grid_cells_gdf):
    """Convert GeoDataFrame to EE FeatureCollection."""
    features = []
    for _, row in grid_cells_gdf.iterrows():
        geom = ee.Geometry.Rectangle([
            float(row['lon_min']), float(row['lat_min']),
            float(row['lon_max']), float(row['lat_max'])
        ])
        features.append(ee.Feature(geom, {
            'cell_id': int(row['cell_id']),
            'lat_center': float(row['lat_center']),
            'lon_center': float(row['lon_center'])
        }))
    return ee.FeatureCollection(features)

# Create the grid FeatureCollection once
grid_fc = create_grid_feature_collection(active_cells)
print(f"Created FeatureCollection with {len(active_cells)} grid cells")

# GCS output path for CH4 exports
CH4_EXPORT_PREFIX = f'gs://{BUCKET_NAME}/thermokarst_lakes/ch4_weekly'
print(f"Export destination: {CH4_EXPORT_PREFIX}/")

In [ ]:
# Submit GEE export tasks for weekly CH4 extraction
# Each task exports one year-week of CH4 data to GCS

# Maximum uncertainty threshold (ppb) - filter out high-uncertainty retrievals
# Set to None to skip uncertainty filtering
MAX_UNCERTAINTY_PPB = 30  # Typical uncertainty is ~10-20 ppb

def submit_weekly_ch4_export(year, week, grid_fc, max_uncertainty=MAX_UNCERTAINTY_PPB):
    """
    Submit a GEE export task for weekly CH4 extraction.
    
    Uses bias-corrected CH4 band from L3 product (already quality-filtered).
    Optionally filters by uncertainty threshold.
    
    Returns:
    --------
    task : ee.batch.Task
    task_name : str
    """
    # Convert week to date range
    jan4 = datetime(year, 1, 4)
    monday_week1 = jan4 - timedelta(days=jan4.weekday())
    week_start = monday_week1 + timedelta(weeks=week - 1)
    week_end = week_start + timedelta(days=6)
    
    start_date = week_start.strftime('%Y-%m-%d')
    end_date = week_end.strftime('%Y-%m-%d')
    doy_midpoint = (week_start + timedelta(days=3)).timetuple().tm_yday
    
    # Load TROPOMI CH4 collection
    # L3 data is already quality-filtered (qa > 50%) during L2→L3 conversion
    tropomi = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_CH4')
               .filterDate(start_date, end_date)
               .filterBounds(study_area))
    
    # Select bias-corrected CH4 band (recommended for analysis)
    ch4_band = 'CH4_column_volume_mixing_ratio_dry_air_bias_corrected'
    uncertainty_band = 'CH4_column_volume_mixing_ratio_dry_air_uncertainty'
    
    # Apply uncertainty filter if specified
    if max_uncertainty is not None:
        def filter_by_uncertainty(img):
            ch4 = img.select(ch4_band)
            unc = img.select(uncertainty_band)
            mask = unc.lt(max_uncertainty)
            return ch4.updateMask(mask)
        filtered = tropomi.map(filter_by_uncertainty)
    else:
        filtered = tropomi.select(ch4_band)
    
    # Compute weekly mean
    weekly_ch4 = filtered.mean()
    
    # Extract CH4 for all grid cells
    def extract_ch4(feature):
        stats = weekly_ch4.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=feature.geometry(),
            scale=7000,
            maxPixels=1e9
        )
        return feature.set({
            'year': year,
            'week': week,
            'doy': doy_midpoint,
            'ch4_ppb': stats.get(ch4_band),
            'start_date': start_date,
            'end_date': end_date
        })
    
    results = grid_fc.map(extract_ch4)
    
    # Create export task
    task_name = f'ch4_weekly_{year}_w{week:02d}'
    file_name = f'ch4_{year}_w{week:02d}'
    
    task = ee.batch.Export.table.toCloudStorage(
        collection=results,
        description=task_name,
        bucket=BUCKET_NAME,
        fileNamePrefix=f'thermokarst_lakes/ch4_weekly/{file_name}',
        fileFormat='CSV',
        selectors=['cell_id', 'lat_center', 'lon_center', 'year', 'week', 'doy', 'ch4_ppb', 'start_date', 'end_date']
    )
    
    return task, task_name

# Submit all export tasks
print("Submitting GEE export tasks for weekly CH4 extraction...")
print(f"Using bias-corrected CH4 band (L3 data is already qa > 50% filtered)")
if MAX_UNCERTAINTY_PPB:
    print(f"Additional filter: uncertainty < {MAX_UNCERTAINTY_PPB} ppb")
print(f"Years: {YEARS}")
print(f"Weeks: {WEEK_START}-{WEEK_END} ({WEEK_END - WEEK_START + 1} weeks per year)")
print(f"Total tasks to submit: {len(YEARS) * (WEEK_END - WEEK_START + 1)}")
print()

submitted_tasks = []
for year in YEARS:
    print(f"\n=== {year} ===")
    for week in range(WEEK_START, WEEK_END + 1):
        task, task_name = submit_weekly_ch4_export(year, week, grid_fc, max_uncertainty=MAX_UNCERTAINTY_PPB)
        task.start()
        submitted_tasks.append({'year': year, 'week': week, 'task_name': task_name, 'task_id': task.id})
        print(f"  Submitted: {task_name} (ID: {task.id})")

print(f"\n{'='*60}")
print(f"Submitted {len(submitted_tasks)} export tasks")
print(f"Monitor progress at: https://code.earthengine.google.com/tasks")
print(f"Output location: gs://{BUCKET_NAME}/thermokarst_lakes/ch4_weekly/")

# Save task list for reference
tasks_df = pd.DataFrame(submitted_tasks)
tasks_df.to_csv('./ch4_export_tasks.csv', index=False)
print(f"\nTask list saved to: ./ch4_export_tasks.csv")

In [ ]:
# Load exported CH4 data from GCS and build panel dataset
# Run this cell AFTER GEE export tasks complete

print("Loading CH4 data from GCS exports...")

# List all exported CSV files
import subprocess

ch4_files = subprocess.run(
    ['gsutil', 'ls', f'gs://{BUCKET_NAME}/thermokarst_lakes/ch4_weekly/ch4_*.csv'],
    capture_output=True, text=True
).stdout.strip().split('\n')

ch4_files = [f for f in ch4_files if f.endswith('.csv')]
print(f"Found {len(ch4_files)} CH4 export files")

if len(ch4_files) == 0:
    print("\nNo CH4 exports found. Waiting for GEE tasks to complete.")
    print("Monitor progress at: https://code.earthengine.google.com/tasks")
    print("\nRe-run this cell after exports finish.")
    ch4_df = pd.DataFrame()
else:
    # Load and concatenate all CH4 files
    ch4_dfs = []
    for f in ch4_files:
        try:
            df = pd.read_csv(f)
            ch4_dfs.append(df)
        except Exception as e:
            print(f"  Error loading {f}: {e}")
    
    ch4_df = pd.concat(ch4_dfs, ignore_index=True)
    
    # Clean up: remove rows with null CH4 (weeks with no TROPOMI data)
    ch4_df = ch4_df[ch4_df['ch4_ppb'].notna()].copy()
    
    print(f"\nLoaded {len(ch4_df):,} CH4 observations")
    print(f"Unique pixels: {ch4_df['cell_id'].nunique()}")
    print(f"Year range: {ch4_df['year'].min()}-{ch4_df['year'].max()}")
    print(f"Week range: {ch4_df['week'].min()}-{ch4_df['week'].max()}")
    print(f"CH4 range: {ch4_df['ch4_ppb'].min():.1f} - {ch4_df['ch4_ppb'].max():.1f} ppb")
    
    # Show coverage by year
    print(f"\nObservations by year:")
    for year in sorted(ch4_df['year'].unique()):
        n = len(ch4_df[ch4_df['year'] == year])
        print(f"  {year}: {n:,}")
    
    display(ch4_df.head())

In [ ]:
# Merge CH4 data with thaw time series to create panel dataset
print("Building panel dataset...")

if len(ch4_df) == 0:
    print("No CH4 data available. Cannot build panel dataset.")
    panel_df = pd.DataFrame()
else:
    # Ensure cell_id is integer for merge
    ch4_df['cell_id'] = ch4_df['cell_id'].astype(int)
    thaw_df['cell_id'] = thaw_df['cell_id'].astype(int)
    
    # Merge on cell_id, year, week
    panel_df = thaw_df.merge(
        ch4_df[['cell_id', 'year', 'week', 'ch4_ppb']],
        on=['cell_id', 'year', 'week'],
        how='left'
    )
    
    # Count observations with CH4 data
    n_with_ch4 = panel_df['ch4_ppb'].notna().sum()
    n_total = len(panel_df)
    pct_coverage = 100 * n_with_ch4 / n_total
    
    print(f"\nPanel Dataset Summary:")
    print(f"  Total pixel-week observations: {n_total:,}")
    print(f"  Observations with CH4 data: {n_with_ch4:,} ({pct_coverage:.1f}%)")
    print(f"  Missing CH4: {n_total - n_with_ch4:,} ({100 - pct_coverage:.1f}%)")
    print(f"  Unique pixels: {panel_df['cell_id'].nunique()}")
    print(f"  Unique year-weeks: {panel_df.groupby(['year', 'week']).ngroups}")
    
    # Show CH4 coverage by year
    print(f"\nCH4 Coverage by Year:")
    for year in YEARS:
        year_data = panel_df[panel_df['year'] == year]
        n_with = year_data['ch4_ppb'].notna().sum()
        pct = 100 * n_with / len(year_data) if len(year_data) > 0 else 0
        print(f"  {year}: {n_with:,} / {len(year_data):,} ({pct:.1f}%)")
    
    # Create date column for panel index
    panel_df['date'] = panel_df.apply(
        lambda row: datetime(int(row['year']), 1, 4) + timedelta(weeks=int(row['week']) - 1, days=3),
        axis=1
    )
    
    display(panel_df.head(10))

## Part 5: Panel Regression with Fixed Effects

Estimate the relationship between cumulative thawed lake area/perimeter and CH4 using panel regression with pixel and time fixed effects:

**Model:** CH4_it = β₁(thawed_area_it) + α_i + γ_t + ε_it

Where:
- i = pixel (cell_id)
- t = time (year-week)
- α_i = pixel fixed effects (controls for static pixel characteristics like elevation, land cover)
- γ_t = time fixed effects (controls for regional CH4 trends, seasonality)
- β₁ = effect of thawed lake area on CH4 (our parameter of interest)

In [ ]:
# Panel regression analysis
print("="*70)
print("PANEL REGRESSION WITH FIXED EFFECTS")
print("="*70)

if len(panel_df) == 0 or panel_df['ch4_ppb'].notna().sum() < 100:
    print("\nInsufficient data for panel regression.")
else:
    # Filter to observations with CH4 data
    reg_data = panel_df[panel_df['ch4_ppb'].notna()].copy()
    print(f"\nObservations for regression: {len(reg_data):,}")
    print(f"Unique pixels: {reg_data['cell_id'].nunique()}")
    print(f"Unique time periods: {reg_data.groupby(['year', 'week']).ngroups}")
    
    # Create time index for fixed effects (year-week combination)
    reg_data['time_id'] = reg_data['year'] * 100 + reg_data['week']
    
    # Set up panel index
    reg_data = reg_data.set_index(['cell_id', 'time_id'])
    
    # Summary statistics for key variables
    print(f"\nVariable Statistics:")
    print(f"  CH4 (ppb):          {reg_data['ch4_ppb'].mean():.1f} ± {reg_data['ch4_ppb'].std():.1f}")
    print(f"  Thawed Area (km²):  {reg_data['thawed_area_km2'].mean():.2f} ± {reg_data['thawed_area_km2'].std():.2f}")
    print(f"  Thawed Perim (km):  {reg_data['thawed_perim_km'].mean():.2f} ± {reg_data['thawed_perim_km'].std():.2f}")
    print(f"  Frac Thawed:        {reg_data['frac_thawed'].mean():.2f} ± {reg_data['frac_thawed'].std():.2f}")
    
    # ========================================================================
    # Model 1: Thawed Area Only
    # ========================================================================
    print(f"\n{'='*70}")
    print("MODEL 1: Thawed Area Only")
    print(f"{'='*70}")
    
    try:
        mod1 = PanelOLS(
            reg_data['ch4_ppb'],
            reg_data[['thawed_area_km2']],
            entity_effects=True,   # pixel fixed effects
            time_effects=True      # time fixed effects
        )
        results1 = mod1.fit(cov_type='clustered', cluster_entity=True)
        print(results1.summary)
        
        # Store results
        beta_area = results1.params['thawed_area_km2']
        se_area = results1.std_errors['thawed_area_km2']
        pval_area = results1.pvalues['thawed_area_km2']
        r2_within_1 = results1.rsquared_within
        
        print(f"\nKey Result:")
        print(f"  β(thawed_area) = {beta_area:.4f} ppb per km²")
        print(f"  SE = {se_area:.4f}, p = {pval_area:.4f}")
        print(f"  95% CI: [{beta_area - 1.96*se_area:.4f}, {beta_area + 1.96*se_area:.4f}]")
        print(f"  R² within: {r2_within_1:.4f}")
    except Exception as e:
        print(f"Model 1 failed: {e}")
        results1 = None
    
    # ========================================================================
    # Model 2: Thawed Perimeter Only
    # ========================================================================
    print(f"\n{'='*70}")
    print("MODEL 2: Thawed Perimeter Only")
    print(f"{'='*70}")
    
    try:
        mod2 = PanelOLS(
            reg_data['ch4_ppb'],
            reg_data[['thawed_perim_km']],
            entity_effects=True,
            time_effects=True
        )
        results2 = mod2.fit(cov_type='clustered', cluster_entity=True)
        print(results2.summary)
        
        beta_perim = results2.params['thawed_perim_km']
        se_perim = results2.std_errors['thawed_perim_km']
        pval_perim = results2.pvalues['thawed_perim_km']
        r2_within_2 = results2.rsquared_within
        
        print(f"\nKey Result:")
        print(f"  β(thawed_perim) = {beta_perim:.4f} ppb per km")
        print(f"  SE = {se_perim:.4f}, p = {pval_perim:.4f}")
        print(f"  95% CI: [{beta_perim - 1.96*se_perim:.4f}, {beta_perim + 1.96*se_perim:.4f}]")
        print(f"  R² within: {r2_within_2:.4f}")
    except Exception as e:
        print(f"Model 2 failed: {e}")
        results2 = None
    
    # ========================================================================
    # Model 3: Both Predictors
    # ========================================================================
    print(f"\n{'='*70}")
    print("MODEL 3: Both Thawed Area and Perimeter")
    print(f"{'='*70}")
    
    try:
        mod3 = PanelOLS(
            reg_data['ch4_ppb'],
            reg_data[['thawed_area_km2', 'thawed_perim_km']],
            entity_effects=True,
            time_effects=True
        )
        results3 = mod3.fit(cov_type='clustered', cluster_entity=True)
        print(results3.summary)
    except Exception as e:
        print(f"Model 3 failed: {e}")
        results3 = None

In [ ]:
# Robustness checks
print("="*70)
print("ROBUSTNESS CHECKS")
print("="*70)

if len(panel_df) == 0 or panel_df['ch4_ppb'].notna().sum() < 100:
    print("\nInsufficient data for robustness checks.")
else:
    # ========================================================================
    # Robustness 1: Fractional Thaw (instead of absolute area)
    # ========================================================================
    print(f"\n{'='*70}")
    print("ROBUSTNESS 1: Fractional Thaw")
    print(f"{'='*70}")
    
    try:
        mod_frac = PanelOLS(
            reg_data['ch4_ppb'],
            reg_data[['frac_thawed']],
            entity_effects=True,
            time_effects=True
        )
        results_frac = mod_frac.fit(cov_type='clustered', cluster_entity=True)
        
        beta_frac = results_frac.params['frac_thawed']
        se_frac = results_frac.std_errors['frac_thawed']
        pval_frac = results_frac.pvalues['frac_thawed']
        
        print(f"\nResult:")
        print(f"  β(frac_thawed) = {beta_frac:.4f} ppb per unit fraction")
        print(f"  SE = {se_frac:.4f}, p = {pval_frac:.4f}")
        print(f"  95% CI: [{beta_frac - 1.96*se_frac:.4f}, {beta_frac + 1.96*se_frac:.4f}]")
        print(f"  Interpretation: Going from 0% to 100% thawed changes CH4 by {beta_frac:.2f} ppb")
    except Exception as e:
        print(f"Fractional thaw model failed: {e}")
    
    # ========================================================================
    # Robustness 2: Thaw Season Only (DOY 120-220)
    # ========================================================================
    print(f"\n{'='*70}")
    print("ROBUSTNESS 2: Thaw Season Only (DOY 120-220)")
    print(f"{'='*70}")
    
    try:
        thaw_season = reg_data[(reg_data['doy'] >= 120) & (reg_data['doy'] <= 220)]
        print(f"Observations in thaw season: {len(thaw_season):,}")
        
        if len(thaw_season) >= 100:
            mod_season = PanelOLS(
                thaw_season['ch4_ppb'],
                thaw_season[['thawed_area_km2']],
                entity_effects=True,
                time_effects=True
            )
            results_season = mod_season.fit(cov_type='clustered', cluster_entity=True)
            
            beta_season = results_season.params['thawed_area_km2']
            se_season = results_season.std_errors['thawed_area_km2']
            pval_season = results_season.pvalues['thawed_area_km2']
            
            print(f"\nResult (thaw season only):")
            print(f"  β(thawed_area) = {beta_season:.4f} ppb per km²")
            print(f"  SE = {se_season:.4f}, p = {pval_season:.4f}")
            print(f"  95% CI: [{beta_season - 1.96*se_season:.4f}, {beta_season + 1.96*se_season:.4f}]")
        else:
            print("Insufficient observations in thaw season.")
    except Exception as e:
        print(f"Thaw season model failed: {e}")
    
    # ========================================================================
    # Robustness 3: Lagged Effect (1-week lag)
    # ========================================================================
    print(f"\n{'='*70}")
    print("ROBUSTNESS 3: Lagged Effect (CH4 responds 1 week after thaw)")
    print(f"{'='*70}")
    
    try:
        # Create lagged thaw variable
        reg_data_reset = reg_data.reset_index()
        reg_data_reset = reg_data_reset.sort_values(['cell_id', 'year', 'week'])
        reg_data_reset['thawed_area_lag1'] = reg_data_reset.groupby('cell_id')['thawed_area_km2'].shift(1)
        
        # Filter to valid lagged observations
        lagged = reg_data_reset[reg_data_reset['thawed_area_lag1'].notna()].copy()
        lagged = lagged.set_index(['cell_id', 'time_id'])
        
        print(f"Observations with lagged data: {len(lagged):,}")
        
        if len(lagged) >= 100:
            mod_lag = PanelOLS(
                lagged['ch4_ppb'],
                lagged[['thawed_area_lag1']],
                entity_effects=True,
                time_effects=True
            )
            results_lag = mod_lag.fit(cov_type='clustered', cluster_entity=True)
            
            beta_lag = results_lag.params['thawed_area_lag1']
            se_lag = results_lag.std_errors['thawed_area_lag1']
            pval_lag = results_lag.pvalues['thawed_area_lag1']
            
            print(f"\nResult (1-week lagged thaw):")
            print(f"  β(thawed_area_lag1) = {beta_lag:.4f} ppb per km²")
            print(f"  SE = {se_lag:.4f}, p = {pval_lag:.4f}")
            print(f"  95% CI: [{beta_lag - 1.96*se_lag:.4f}, {beta_lag + 1.96*se_lag:.4f}]")
        else:
            print("Insufficient observations with lagged data.")
    except Exception as e:
        print(f"Lagged model failed: {e}")

## Part 6: Visualization & Diagnostics

Generate publication-quality figures for the panel regression results.

In [ ]:
# Figure 1: Coefficient plot comparing models
if len(panel_df) > 0 and panel_df['ch4_ppb'].notna().sum() >= 100:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Collect results from all models
    model_results = []
    
    if 'results1' in dir() and results1 is not None:
        model_results.append({
            'model': 'Model 1:\nThawed Area',
            'beta': results1.params['thawed_area_km2'],
            'se': results1.std_errors['thawed_area_km2'],
            'predictor': 'thawed_area_km2'
        })
    
    if 'results2' in dir() and results2 is not None:
        model_results.append({
            'model': 'Model 2:\nThawed Perimeter',
            'beta': results2.params['thawed_perim_km'],
            'se': results2.std_errors['thawed_perim_km'],
            'predictor': 'thawed_perim_km'
        })
    
    if 'results_frac' in dir() and results_frac is not None:
        model_results.append({
            'model': 'Robustness 1:\nFrac. Thawed',
            'beta': results_frac.params['frac_thawed'],
            'se': results_frac.std_errors['frac_thawed'],
            'predictor': 'frac_thawed'
        })
    
    if 'results_season' in dir() and results_season is not None:
        model_results.append({
            'model': 'Robustness 2:\nThaw Season Only',
            'beta': results_season.params['thawed_area_km2'],
            'se': results_season.std_errors['thawed_area_km2'],
            'predictor': 'thawed_area_km2'
        })
    
    if 'results_lag' in dir() and results_lag is not None:
        model_results.append({
            'model': 'Robustness 3:\n1-Week Lag',
            'beta': results_lag.params['thawed_area_lag1'],
            'se': results_lag.std_errors['thawed_area_lag1'],
            'predictor': 'thawed_area_lag1'
        })
    
    if model_results:
        results_summary = pd.DataFrame(model_results)
        
        # Plot
        y_pos = range(len(results_summary))
        
        for i, row in results_summary.iterrows():
            ci_low = row['beta'] - 1.96 * row['se']
            ci_high = row['beta'] + 1.96 * row['se']
            
            # Color based on significance
            color = 'tab:blue' if abs(row['beta']) > 1.96 * row['se'] else 'gray'
            
            ax.errorbar(row['beta'], i, xerr=1.96 * row['se'],
                       fmt='o', color=color, capsize=5, capthick=2, markersize=10)
        
        ax.axvline(x=0, color='black', linestyle='--', linewidth=1, alpha=0.7)
        ax.set_yticks(range(len(results_summary)))
        ax.set_yticklabels(results_summary['model'])
        ax.set_xlabel('Coefficient (ppb per unit predictor)')
        ax.set_title('Panel Regression Results: Effect of Thaw on CH4\n(Pixel and Time Fixed Effects, Clustered SEs)')
        ax.grid(True, alpha=0.3, axis='x')
        
        plt.tight_layout()
        plt.savefig(f'{FIGURES_DIR}/manuscript/ch4_panel_regression_results.png', dpi=300, bbox_inches='tight')
        plt.show()
        print("Saved: ch4_panel_regression_results.png")
    else:
        print("No model results available for plotting.")
else:
    print("Insufficient data for coefficient plot.")

In [ ]:
# Figure 2: CH4 vs Thawed Area scatter (residualized)
if len(panel_df) > 0 and panel_df['ch4_ppb'].notna().sum() >= 100:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Get residualized data (demean by cell and time)
    reg_data_plot = panel_df[panel_df['ch4_ppb'].notna()].copy()
    
    # Simple demeaning for visualization
    reg_data_plot['ch4_demeaned'] = reg_data_plot.groupby('cell_id')['ch4_ppb'].transform(lambda x: x - x.mean())
    reg_data_plot['thaw_demeaned'] = reg_data_plot.groupby('cell_id')['thawed_area_km2'].transform(lambda x: x - x.mean())
    
    # Panel A: Raw scatter
    ax = axes[0]
    ax.scatter(reg_data_plot['thawed_area_km2'], reg_data_plot['ch4_ppb'], 
               alpha=0.3, s=10, c='tab:blue')
    ax.set_xlabel('Thawed Area (km²)')
    ax.set_ylabel('CH4 (ppb)')
    ax.set_title('Raw Data')
    ax.grid(True, alpha=0.3)
    
    # Add correlation
    valid = reg_data_plot[['thawed_area_km2', 'ch4_ppb']].dropna()
    r, p = stats.pearsonr(valid['thawed_area_km2'], valid['ch4_ppb'])
    ax.text(0.05, 0.95, f'r = {r:.3f}\np = {p:.2e}', transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Panel B: Within-pixel variation (demeaned)
    ax = axes[1]
    ax.scatter(reg_data_plot['thaw_demeaned'], reg_data_plot['ch4_demeaned'], 
               alpha=0.3, s=10, c='tab:green')
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    ax.axvline(x=0, color='black', linestyle='--', alpha=0.5)
    ax.set_xlabel('Thawed Area (km², demeaned by pixel)')
    ax.set_ylabel('CH4 (ppb, demeaned by pixel)')
    ax.set_title('Within-Pixel Variation\n(Pixel means removed)')
    ax.grid(True, alpha=0.3)
    
    # Add correlation
    valid = reg_data_plot[['thaw_demeaned', 'ch4_demeaned']].dropna()
    r, p = stats.pearsonr(valid['thaw_demeaned'], valid['ch4_demeaned'])
    ax.text(0.05, 0.95, f'r = {r:.3f}\np = {p:.2e}', transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.suptitle('CH4 vs Thawed Lake Area', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/supplementary/ch4_thaw_scatter.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Saved: ch4_thaw_scatter.png")
else:
    print("Insufficient data for scatter plots.")

In [ ]:
# Figure 3: CH4 data coverage heatmap
if len(panel_df) > 0:
    # Create coverage matrix
    coverage = panel_df.pivot_table(
        index='cell_id',
        columns=['year', 'week'],
        values='ch4_ppb',
        aggfunc='count'
    ).fillna(0)
    
    # Summarize coverage by week
    week_coverage = panel_df.groupby(['year', 'week']).agg({
        'ch4_ppb': lambda x: x.notna().sum(),
        'cell_id': 'nunique'
    }).reset_index()
    week_coverage.columns = ['year', 'week', 'n_obs', 'n_cells']
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Panel A: Coverage by week over all years
    ax = axes[0]
    for year in YEARS:
        year_data = week_coverage[week_coverage['year'] == year]
        ax.plot(year_data['week'], year_data['n_obs'], 'o-', label=str(year), alpha=0.7)
    
    ax.set_xlabel('ISO Week')
    ax.set_ylabel('Number of CH4 Observations')
    ax.set_title('TROPOMI CH4 Data Availability by Week')
    ax.legend(title='Year')
    ax.grid(True, alpha=0.3)
    
    # Panel B: Missing data pattern
    ax = axes[1]
    missing_pct = panel_df.groupby(['year', 'week'])['ch4_ppb'].apply(lambda x: 100 * x.isna().mean())
    missing_df = missing_pct.reset_index()
    missing_df.columns = ['year', 'week', 'pct_missing']
    
    for year in YEARS:
        year_data = missing_df[missing_df['year'] == year]
        ax.plot(year_data['week'], year_data['pct_missing'], 'o-', label=str(year), alpha=0.7)
    
    ax.set_xlabel('ISO Week')
    ax.set_ylabel('% Missing CH4')
    ax.set_title('Missing Data Pattern by Week')
    ax.legend(title='Year')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 100)
    
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/supplementary/ch4_data_coverage.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: ch4_data_coverage.png")
else:
    print("No panel data for coverage plot.")

## Part 7: Export Results

Export panel dataset and regression results to GCS.

In [ ]:
# Export panel dataset
print("Exporting results...")

if len(panel_df) == 0:
    print("No panel data to export.")
else:
    # Export panel dataset
    export_cols = [
        'cell_id', 'year', 'week', 'doy', 'date',
        'ch4_ppb', 'n_ch4_obs',
        'thawed_area_km2', 'thawed_perim_km', 'frac_thawed', 'n_thawed_lakes',
        'total_area_km2', 'total_perim_km', 'n_lakes',
        'lat_center', 'lon_center'
    ]
    
    # Filter to available columns
    available_cols = [c for c in export_cols if c in panel_df.columns]
    panel_export = panel_df[available_cols].copy()
    
    # Convert date to string for CSV
    if 'date' in panel_export.columns:
        panel_export['date'] = panel_export['date'].astype(str)
    
    panel_path = f'{RESULTS_GCS}/ch4_panel_data_weekly_2019-2023.csv'
    try:
        panel_export.to_csv(panel_path, index=False)
        print(f"  Exported: {panel_path}")
        print(f"  Rows: {len(panel_export):,}")
    except Exception as e:
        print(f"  GCS export failed: {e}")
        local_path = './ch4_panel_data_weekly_2019-2023.csv'
        panel_export.to_csv(local_path, index=False)
        print(f"  Saved locally: {local_path}")

# Export regression summary
if 'model_results' in dir() and model_results:
    results_path = f'{RESULTS_GCS}/ch4_panel_regression_summary.csv'
    try:
        pd.DataFrame(model_results).to_csv(results_path, index=False)
        print(f"  Exported: {results_path}")
    except Exception as e:
        print(f"  GCS export failed: {e}")
        local_path = './ch4_panel_regression_summary.csv'
        pd.DataFrame(model_results).to_csv(local_path, index=False)
        print(f"  Saved locally: {local_path}")

print("\nExport complete!")

## Summary

This notebook analyzed the relationship between TROPOMI CH4 column concentrations and cumulative lake thaw on Alaska's North Slope using panel regression with fixed effects.

**Scientific Question:** Does lake ice-off timing predict TROPOMI methane concentrations?

**Approach:**
- Weekly temporal resolution (ISO weeks 9-44, ~35 weeks Mar-Oct)
- Panel regression: CH4_it = β₁(thawed_area_it) + pixel_FE_i + week_FE_t + ε_it
- TROPOMI QA filtering: qa_value > 0.5
- Clustered standard errors by pixel

**Key Tests:**
1. **Model 1:** Thawed area as sole predictor
2. **Model 2:** Thawed perimeter as sole predictor
3. **Model 3:** Both area and perimeter
4. **Robustness 1:** Fractional thaw (0-1 scale)
5. **Robustness 2:** Thaw season only (DOY 120-220)
6. **Robustness 3:** 1-week lagged effects

**Output Files:**
- `ch4_panel_data_weekly_2019-2023.csv`: Full panel dataset
- `ch4_panel_regression_summary.csv`: Model coefficients and standard errors

**Figures Generated:**
- `ch4_analysis_grid.png`: Study area grid with lake density
- `ch4_thaw_progression.png`: Cumulative thaw curves by year
- `ch4_panel_regression_results.png`: Coefficient plot comparing models
- `ch4_thaw_scatter.png`: CH4 vs thawed area (raw and demeaned)
- `ch4_data_coverage.png`: TROPOMI data availability patterns